# 3. Analyze accuracy and efficiency

This notebook consumes generated artifacts. It does not copy numeric results into notebook state, so rerunning a preset automatically updates the analysis.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from vision_bench.config import load_project_config
from vision_bench.reporting import aggregate_results, collect_run_results, paired_distillation_gains

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
PRESET = "quick"  # change to 'full' only for a completed reportable suite
project = load_project_config(PRESET, PROJECT_ROOT)

## Accuracy table and missing runs

In [ ]:
rows, histories, missing = collect_run_results(project, ARTIFACT_ROOT)
print(f"Completed: {len(rows)} / {len(project.runs)}; missing: {missing}")
pd.DataFrame(aggregate_results(rows)) if rows else pd.DataFrame()

## Paired hard-distillation gain

Only matched Distilled DeiT seeds enter this table.

In [ ]:
pd.DataFrame(paired_distillation_gains(rows))

## Hardware measurements

The same-device benchmark must exist before interpreting throughput. Filter by one batch size and precision before ranking models.

In [ ]:
benchmark_path = ARTIFACT_ROOT / PRESET / "benchmark.json"
if benchmark_path.exists():
    benchmark = json.loads(benchmark_path.read_text())
    timings = pd.DataFrame(benchmark["measurements"])
    display(timings[(timings["status"] == "complete") & (timings["batch_size"] == 64)])
else:
    print("Run vision-bench benchmark for this preset first.")

## Create the publication artifacts

Run `uv run vision-bench report --preset quick` (or `full`) from a terminal. Open `artifacts/<preset>/report/report.md` and use its linked CSV files for any custom analysis.